# 14. Stacking with a Random Forest meta-learner

Trains a Random Forest on the per-pixel binary outputs of the four base
models (validation split) and evaluates it on the held-out test split.
Serves as the stacking baseline for the ensemble comparison: with binary
inputs of four models the meta-learner observes at most 2^4 = 16 distinct
feature combinations, over which it can in principle represent any Boolean rule; its underperformance is therefore driven not by the functional class but by the discarded probabilities and confidences, the pixel-independent fitting dominated by the largest scenes, and the balanced class weighting at an object share of about 3%.

In [ ]:
# ============================================================================
# CONFIGURATION
# ============================================================================

import os
import re
import glob
import pickle
import numpy as np
import cv2
from pathlib import Path
from tqdm import tqdm
from sklearn.ensemble import RandomForestClassifier

# Binary masks of the base models, produced by notebooks 02, 05, 10, 12
# (validation split is used for meta-training, test split for evaluation)
VAL_MODEL_DIRS = {
    "YOLO":           "val_masks/yolo",
    "dlv3+-ResNet50": "val_masks/dlv3+_with_resnet50",
    "Mask2Former":    "val_masks/mask2former",
    "UNet-ResNet50":  "val_masks/unet_with_resnet50",
}
VAL_GT_DIR = "split_data/val/Masks"

TEST_MODEL_DIRS = {
    "YOLO":           "test_masks/yolo",
    "dlv3+-ResNet50": "test_masks/dlv3+_with_resnet50",
    "Mask2Former":    "test_masks/mask2former",
    "UNet-ResNet50":  "test_masks/unet_with_resnet50",
}
TEST_GT_DIR = "split_data/test/Masks"

MODEL_SAVE_PATH = "rf_stacking_model.pkl"

# Validation Dice of the base models (used only for the reference
# weighted-averaging ensemble reported in the comparison table)
VALIDATION_DICE_SCORES = {
    "YOLO": None,
    "dlv3+-ResNet50": None,
    "Mask2Former": None,
    "UNet-ResNet50": None,
}

# Random Forest hyperparameters (single configuration, no tuning)
RF_PARAMS = {
    "n_estimators": 50,
    "max_depth": 10,
    "min_samples_split": 2,
    "min_samples_leaf": 1,
    "max_features": "sqrt",
    "random_state": 42,
    "n_jobs": -1,
    "class_weight": "balanced",
}

MODEL_ORDER = list(VAL_MODEL_DIRS.keys())

In [ ]:
# ============================================================================
# HELPER FUNCTIONS
# ============================================================================

def match_filename(pred_name, gt_names):
    """Matches a prediction file name to the corresponding GT file name."""
    base_name = pred_name.replace("_pred.png", ".png")
    if base_name in gt_names:
        return base_name
    m = re.search(r"(\d{4}-\d{2}-\d{2})", pred_name)
    if m:
        date = m.group(1)
        for gt_name in gt_names:
            if date in gt_name:
                return gt_name
    if pred_name in gt_names:
        return pred_name
    return None


def load_binary(path, ref_shape=None):
    """Loads a mask as a binary {0, 1} array, resizing to ref_shape if given."""
    mask = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if mask is None:
        return None
    if ref_shape is not None and mask.shape != ref_shape:
        mask = cv2.resize(mask, (ref_shape[1], ref_shape[0]),
                          interpolation=cv2.INTER_NEAREST)
    return (mask > 127).astype(np.uint8)


def dice_score(pred, gt):
    """Dice coefficient between two binary masks."""
    inter = np.logical_and(pred, gt).sum()
    denom = pred.sum() + gt.sum()
    if denom == 0:
        return 1.0
    return 2.0 * inter / denom


def collect_scene_features(gt_dir, model_dirs):
    """Yields (name, gt_binary, feature_stack) per scene.

    feature_stack has shape (H, W, n_models) with binary predictions of the
    base models resized to the GT geometry.
    """
    gt_files = sorted(glob.glob(os.path.join(gt_dir, "*.png")))
    gt_names = [os.path.basename(f) for f in gt_files]
    for gt_path in gt_files:
        gt = load_binary(gt_path)
        if gt is None:
            continue
        name = os.path.basename(gt_path)
        stack, complete = [], True
        for model_name in MODEL_ORDER:
            mdir = model_dirs[model_name]
            cand = [os.path.join(mdir, name),
                    os.path.join(mdir, name.replace(".png", "_pred.png"))]
            pred = None
            for p in cand:
                if os.path.exists(p):
                    pred = load_binary(p, ref_shape=gt.shape)
                    break
            if pred is None:
                complete = False
                break
            stack.append(pred)
        if not complete:
            continue
        yield name, gt, np.stack(stack, axis=-1)

In [ ]:
# ============================================================================
# META-TRAINING ON THE VALIDATION SPLIT
# ============================================================================
# Features: per-pixel binary outputs of the four base models (4 features).
# Target:   per-pixel GT label. The meta-learner never sees the test split.

X_list, y_list = [], []
n_scenes = 0
for name, gt, feats in tqdm(collect_scene_features(VAL_GT_DIR, VAL_MODEL_DIRS),
                            desc="Loading validation scenes"):
    X_list.append(feats.reshape(-1, feats.shape[-1]))
    y_list.append(gt.reshape(-1))
    n_scenes += 1

X = np.concatenate(X_list, axis=0)
y = np.concatenate(y_list, axis=0)
del X_list, y_list

print("Scenes used for meta-training : %d" % n_scenes)
print("Pixels                        : %s" % format(len(y), ","))
print("Positive fraction             : %.1f%%" % (100.0 * y.mean()))

rf = RandomForestClassifier(**RF_PARAMS)
rf.fit(X, y)

print("\nFeature importances:")
for model_name, imp in zip(MODEL_ORDER, rf.feature_importances_):
    print("   %-16s %.4f" % (model_name, imp))

with open(MODEL_SAVE_PATH, "wb") as f:
    pickle.dump(rf, f)
print("\nModel saved to: %s" % MODEL_SAVE_PATH)

In [ ]:
# ============================================================================
# EVALUATION ON THE TEST SPLIT
# ============================================================================
# Per-image Dice of the meta-model, the base models and the reference
# Dice-weighted probability averaging ensemble (notebook 13).

per_image = {m: [] for m in MODEL_ORDER}
per_image["RF stacking"] = []
tp_total = fp_total = fn_total = tn_total = 0

for name, gt, feats in tqdm(collect_scene_features(TEST_GT_DIR, TEST_MODEL_DIRS),
                            desc="Evaluating test scenes"):
    for i, model_name in enumerate(MODEL_ORDER):
        per_image[model_name].append(dice_score(feats[..., i], gt))
    pred = rf.predict(feats.reshape(-1, feats.shape[-1])).reshape(gt.shape)
    per_image["RF stacking"].append(dice_score(pred, gt))
    tp_total += int(np.logical_and(pred == 1, gt == 1).sum())
    fp_total += int(np.logical_and(pred == 1, gt == 0).sum())
    fn_total += int(np.logical_and(pred == 0, gt == 1).sum())
    tn_total += int(np.logical_and(pred == 0, gt == 0).sum())

print("\n%-18s %8s %8s" % ("Model", "Dice", "Std"))
print("-" * 38)
for m in MODEL_ORDER + ["RF stacking"]:
    d = np.array(per_image[m])
    print("%-18s %8.4f %8.4f" % (m, d.mean(), d.std()))

prec = tp_total / max(tp_total + fp_total, 1)
rec = tp_total / max(tp_total + fn_total, 1)
print("\nRF stacking pixel-level confusion matrix (test):")
print("   TP: %s   FP: %s" % (format(tp_total, ","), format(fp_total, ",")))
print("   FN: %s   TN: %s" % (format(fn_total, ","), format(tn_total, ",")))
print("   Precision: %.4f   Recall: %.4f" % (prec, rec))

The meta-learner is trained on the validation split only; the test split is
used exclusively for evaluation. The reported paper values were obtained
with this configuration (random_state = 42).